In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.image import resize
import matplotlib.pyplot as plt

import librosa

data_folder = "../Data/genres_original"
classes = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock']
def preprocessing_wav_to_melspectogram(data_folder, classes, target_shape = (150,150), chunk_duration = 4, overlap_duration = 2):
    melspectograms_list = []
    labels_list = []
    for class_number, class_name in enumerate(classes):
        class_folder = os.path.join(data_folder, class_name)
        print(f'Processing of class {class_name} data is ongoing')
        for filename in os.listdir(class_folder):
            if filename.endswith('.wav'):
                file_path = os.path.join(class_folder, filename)
                audio_data, sample_rate = librosa.load(file_path, sr = None)
                chunk_samples = int(sample_rate * chunk_duration)
                overlap_samples = int(sample_rate * overlap_duration)

                num_of_chunks = int(np.ceil((len(audio_data) - chunk_samples) /(chunk_samples -  overlap_samples)))+1

                for i in range(num_of_chunks):
                    start = i * (chunk_samples - overlap_samples)
                    end = start + chunk_samples

                    chunk = audio_data[start:end]

                    mel_spectogram = librosa.feature.melspectrogram(y = chunk, sr = sample_rate)

                    mel_spectogram = resize(np.expand_dims(mel_spectogram, axis = -1),target_shape)

                    melspectograms_list.append(mel_spectogram)
                    labels_list.append(class_number)
    return np.array(melspectograms_list), np.array(labels_list)
data, labels = preprocessing_wav_to_melspectogram(data_folder, classes)
from tensorflow.keras.utils import to_categorical
labels = to_categorical(labels,num_classes = len(classes))
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(data,labels,test_size=0.2,random_state=42)

Processing of class blues data is ongoing
Processing of class classical data is ongoing
Processing of class country data is ongoing
Processing of class disco data is ongoing
Processing of class hiphop data is ongoing
Processing of class jazz data is ongoing
Processing of class metal data is ongoing
Processing of class pop data is ongoing
Processing of class reggae data is ongoing
Processing of class rock data is ongoing


In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, BatchNormalization, Flatten, Dense, Dropout, Input

# Zdefiniuj kluczowe stałe
INPUT_SHAPE = (150, 150, 1) # Wymiar Twoich spektrogramów (Wysokość, Szerokość, Kanały)
NUM_CLASSES = 10           # Liczba gatunków muzycznych

# Użyj modelu Sequential
second_model = Sequential([
    # Warstwa wejściowa (Input Layer)
    Input(shape=INPUT_SHAPE),

    # Blok 1: Conv1 + Pool1 + BatchNorm1
    # Conv1: 32 filtry, rozmiar jądra 3x3, padding='same'
    Conv2D(filters=32, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu', name='conv1'),
    # BatchNorm1: Normalizacja wsadowa (PyTorch 'bn1')
    BatchNormalization(name='bn1'),
    # Pool1: Redukcja wymiaru 2x2, stride=2
    MaxPool2D(pool_size=(2, 2), strides=(2, 2), name='pool1'),

    # Blok 2: Conv2 + Pool2 + BatchNorm2
    # Conv2: 64 filtry, rozmiar jądra 3x3, padding='same'
    Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu', name='conv2'),
    # BatchNorm2: Normalizacja wsadowa (PyTorch 'bn2')
    BatchNormalization(name='bn2'),
    # Pool2: Redukcja wymiaru 2x2, stride=2
    MaxPool2D(pool_size=(2, 2), strides=(2, 2), name='pool2'),

    # Przekształcanie do wektora dla warstw gęstych
    Flatten(name='flatten'),

    # Warstwy Gęste (Fully Connected)
    # FC1: 128 jednostek (PyTorch 'fc1')
    Dense(units=128, activation='relu', name='fc1'),
    # Dropout: Prawdopodobieństwo 0.5 (PyTorch 'dropout1')
    Dropout(rate=0.5, name='dropout1'),

    # FC2 (Wyjściowa): 10 jednostek, 'softmax' do klasyfikacji (PyTorch 'fc2')
    Dense(units=NUM_CLASSES, activation='softmax', name='fc2')
])

# Wyświetl podsumowanie modelu
second_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 150, 150, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 150, 150, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 75, 75, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 87616)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 128)            │    11,214,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,235,466 (42.86 MB)

 Trainable params: 11,235,274 (42.86 MB)

 Non-trainable params: 192 (768.00 B)

In [6]:
from tensorflow.keras.optimizers import Adam

# Kompilacja modelu
second_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model został zdefiniowany i skompilowany, gotowy do treningu.")

Model został zdefiniowany i skompilowany, gotowy do treningu.


In [ ]:
if 'X_train' in locals() and 'Y_train' in locals():
    print("\nRozpoczynanie treningu (30 epok, batch_size=32)...")
    training_history = second_model.fit(
        X_train,
        Y_train,
        epochs=30,
        batch_size=32,
        validation_data=(X_test, Y_test)
    )
    print("\nTrening zakończony!")
else:
    print("\n--- UWAGA: Dane treningowe (X_train, Y_train) nie są dostępne. ---")
    print("Nie można uruchomić model.fit. Upewnij się, że wykonałeś preprocessing i podział danych (komórki 1-8).")


Rozpoczynanie treningu (30 epok, batch_size=32)...
Epoch 1/30
 32/375 ━━━━━━━━━━━━━━━━━━━━ 5:03 886ms/step - accuracy: 0.2105 - loss: 3.4867